In [28]:
import mlflow
import mlflow.sklearn
import pandas as pd
import re
import string
import numpy as np
import os


from sklearn.metrics import(
    accuracy_score, 
    precision_score,
    recall_score, 
    f1_score
)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression


In [16]:
df =  pd.read_csv("https://raw.githubusercontent.com/Giohanny/Twitter-Sentiment-Analysis/refs/heads/master/text_emotion.csv").drop(columns=['tweet_id','author'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [17]:
# Keep only Happiness and Sadness
df = df[df["sentiment"].isin(["happiness", "sadness"])].copy()

# Encode the labels
df["sentiment"] = df["sentiment"].map({
    "happiness": 1,
    "sadness": 0
})

# Check the result
print(df.head())
print(df["sentiment"].value_counts())

   sentiment                                            content
1          0  Layin n bed with a headache  ughhhh...waitin o...
2          0                Funeral ceremony...gloomy friday...
6          0  I should be sleep, but im not! thinking about ...
8          0            @charviray Charlene my love. I miss you
9          0         @kelcouch I'm sorry  at least it's Friday?
sentiment
1    5209
0    5165
Name: count, dtype: int64


In [18]:
def lemmatization(text):
    lemmatizer= WordNetLemmatizer()

    text = text.split()

    text=[lemmatizer.lemmatize(y) for y in text]

    return " " .join(text)

def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    Text=[i for i in str(text).split() if i not in stop_words]
    return " ".join(Text)

def removing_numbers(text):
    text=''.join([i for i in text if not i.isdigit()])
    return text

def lower_case(text:str):

    text = text.split()

    text=[y.lower() for y in text]

    return " " .join(text)

def removing_punctuations(text):
    ## Remove punctuations
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,،-./:;<=>؟?@[\]^_`{|}~"""), ' ', text)
    text = text.replace('؛',"", )

    ## remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    text =  " ".join(text.split())
    return text.strip()

def removing_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_small_sentences(df):
    for i in range(len(df)):
        if len(df.text.iloc[i].split()) < 3:
            df.text.iloc[i] = np.nan

def normalize_text(df):

    df.content=df.content.apply(lambda content : lower_case(content))
    df.content=df.content.apply(lambda content : remove_stop_words(content))
    df.content=df.content.apply(lambda content : removing_numbers(content))
    df.content=df.content.apply(lambda content : removing_punctuations(content))
    df.content=df.content.apply(lambda content : removing_urls(content))
    df.content=df.content.apply(lambda content : lemmatization(content))
    
    return df


In [19]:
df = normalize_text(df)
df.head()

,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im not thinking old friend want married ...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [ ]:
#  Apply CountVectorizer.
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [21]:
# Splitting the data into train_test_split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
import dagshub

dagshub.init(repo_owner="kush2501", repo_name="mlops-mini-project", mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/kush2501/mlops-mini-project.mlflow")

mlflow.set_experiment("LogisticRegression Baseline..")

Accessing as kush2501

Initialized MLflow to track repo "kush2501/mlops-mini-project"

Repository kush2501/mlops-mini-project initialized!

2026/07/10 15:45:23 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression Baseline..' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/21f7dce6dac245229bd98a48c9fbcc13', creation_time=1783678523706, experiment_id='0', last_update_time=1783678523706, lifecycle_stage='active', name='LogisticRegression Baseline..', tags={}>

In [29]:
with mlflow.start_run():

    # Log preprocessing parameters.
    mlflow.log_param("vectorizer ", "Bag of Words")
    mlflow.log_param("Num of features ", 1000)
    mlflow.log_param("test_size", 0.2)

    # Model building and training.
    model = LogisticRegression()
    model.fit(X_train, y_train)

    # Log model Parameters.
    mlflow.log_param("Model", "Logistic Regression")

    # Model evaluation.
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Log Evaluation matrics.
    mlflow.log_metric("accuracy ", accuracy)
    mlflow.log_metric("precision ", precision)
    mlflow.log_metric("recall ", recall)
    mlflow.log_metric("f1_score",f1)

    # Log model.
    mlflow.sklearn.log_model(model, "model")

    # Save and log the notebook.
    notebook_path = "exp1_baseline.ipynb"
    os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
    mlflow.log_artifact(notebook_path)

    # Print the result for verification.
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"recall: {recall}")
    print(f"f1_score: {f1}")


2026/07/10 15:59:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Accuracy: 0.7773493975903615
Precision: 0.7692307692307693
recall: 0.7783251231527094
f1_score: 0.7737512242899118
🏃 View run hilarious-midge-832 at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/0/runs/dce8745a265a40f9a4853dc848980d47
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/0
